# BPLLM trên Kaggle: chạy LLM và chấm điểm (oracle)

Model: `angeloc1/llama3dot1FoodDel4v05` (Llama 3.1 8B fine-tune cho Food Delivery, của nhóm tác giả bài báo).

**Trước khi chạy, cài đặt ở cột bên phải của notebook:**
1. **Accelerator:** GPU T4 x2
2. **Internet:** On
3. **Input:** Add Input, chọn dataset chứa 2 file `food_delivery_top10.jsonl` và `food_delivery_top10_enriched.jsonl`

Sau đó chạy lần lượt từng khối từ trên xuống.

## Khối 1. Kiểm tra GPU

Phải thấy bảng có Tesla T4. Nếu báo lỗi thì chưa bật Accelerator.

In [ ]:
!nvidia-smi

## Khối 2. Cài thư viện

In [ ]:
!pip install -q "transformers>=4.43" accelerate bitsandbytes

## Khối 3. Tìm và đọc 2 file top-10

`specific` là chunk theo bài báo, `enriched` là chunk cải tiến. Mỗi file phải có 104 câu hỏi.

In [ ]:
import glob
import json

INPUT_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"


def find_input(name):
  matches = glob.glob(f"{INPUT_DIR}/**/{name}", recursive=True)
  assert matches, f"Không thấy {name} trong {INPUT_DIR}. Hãy Add Input dataset chứa file này."
  return matches[0]


def read_jsonl(path):
  with open(path, encoding="utf-8") as f:
    return [json.loads(line) for line in f if line.strip()]


INPUT_FILES = {
    "specific": find_input("food_delivery_top10.jsonl"),
    "enriched": find_input("food_delivery_top10_enriched.jsonl"),
}
for name, path in INPUT_FILES.items():
  print(f"{name}: {len(read_jsonl(path))} câu hỏi, {path}")

## Khối 4. Nạp model (4-bit)

Lần đầu tải khoảng 16 GB nên mất vài phút.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_path = "angeloc1/llama3dot1FoodDel4v05"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True,
                                           bnb_4bit_compute_dtype=torch.float16),
).eval()

## Khối 5. Dựng prompt và hàm hỏi model

Prompt theo đúng Hình 4 của bài báo: câu lệnh, 10 chunk, rồi câu hỏi. Mỗi câu hỏi là một hội thoại mới; model không nhận `expected`.

In [ ]:
PROMPT = ("Use the following pieces of context to answer the question at the end.\n"
          "Answer 'yes' if true or 'no' if false.\n\n"
          "{context}\n\n"
          "Question: {question}")


def build_prompt(record):
  context = "\n\n".join(chunk["text"] for chunk in record["contexts"])
  return PROMPT.format(context=context, question=record["question"])


def ask(record, max_new_tokens=128):
  messages = [{"role": "user", "content": build_prompt(record)}]
  inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                         return_dict=True, return_tensors="pt").to(model.device)
  with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
  return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:],
                          skip_special_tokens=True).strip()

## Khối 6. Thử 1 câu hỏi

Xem prompt và câu trả lời có hợp lý không trước khi chạy cả 104 câu.

In [ ]:
sample = read_jsonl(INPUT_FILES["specific"])[0]
print(build_prompt(sample))
print("-" * 70)
print("Model trả lời:", ask(sample))
print("Đáp án đúng  :", sample["expected"])

## Khối 7. Sinh câu trả lời cho 104 câu, mỗi loại chunk một file

Kết quả ghi vào `/kaggle/working/answers_specific.jsonl` và `answers_enriched.jsonl`.

In [ ]:
from tqdm.auto import tqdm


def run(name):
  output_file = f"{OUTPUT_DIR}/answers_{name}.jsonl"
  with open(output_file, "w", encoding="utf-8") as f:
    for record in tqdm(read_jsonl(INPUT_FILES[name]), desc=name):
      row = {"id": record["id"], "question": record["question"],
             "expected": record["expected"], "answer": ask(record)}
      f.write(json.dumps(row, ensure_ascii=False) + "\n")
  return output_file


ANSWER_FILES = {name: run(name) for name in INPUT_FILES}

## Khối 8. Oracle

Giống `src/bpllm/evaluation/oracle.py`: lấy `yes` hoặc `no` xuất hiện đầu tiên trong câu trả lời rồi so với `expected`. Câu không rút được yes/no tính là sai và được liệt kê riêng.

In [ ]:
import re

_YES_NO = re.compile(r"\b(yes|no)\b", re.IGNORECASE)


def extract_answer(text):
  match = _YES_NO.search(text or "")
  return match.group(1).lower() if match else None


def grade(records):
  total = {"yes": 0, "no": 0}
  correct = {"yes": 0, "no": 0}
  wrong, unparsed = [], []
  for i, record in enumerate(records, 1):
    qid = record.get("id", i)
    expected = str(record["expected"]).strip().lower()
    predicted = extract_answer(record.get("answer", ""))
    total[expected] += 1
    if predicted is None:
      unparsed.append(qid)
    if predicted == expected:
      correct[expected] += 1
    else:
      wrong.append(qid)
  n = len(records)
  return {
      "total": n,
      "correct": sum(correct.values()),
      "accuracy": sum(correct.values()) / n if n else 0.0,
      "accuracy_yes": correct["yes"] / total["yes"] if total["yes"] else 0.0,
      "accuracy_no": correct["no"] / total["no"] if total["no"] else 0.0,
      "wrong_ids": wrong,
      "unparsed_ids": unparsed,
  }


def evaluate(name):
  records = read_jsonl(ANSWER_FILES[name])
  report = grade(records)
  print(f"[{name}] accuracy {report['accuracy']:.2%} ({report['correct']}/{report['total']}), "
        f"yes {report['accuracy_yes']:.2%}, no {report['accuracy_no']:.2%}")
  print(f"  Không rút được yes/no: {report['unparsed_ids']}")
  for record in records:
    if record["id"] in report["wrong_ids"]:
      print(f"  Sai câu {record['id']} (đúng là {record['expected']}): {record['question']}")
      print(f"      Model: {record['answer'][:200]}")
  return report


REPORTS = {}

## Khối 9. Oracle cho chunk theo bài báo (specific)

In [ ]:
REPORTS["specific"] = evaluate("specific")

## Khối 10. Oracle cho chunk enriched

In [ ]:
REPORTS["enriched"] = evaluate("enriched")

## Khối 11. So sánh và lưu báo cáo

File kết quả nằm ở mục **Output** (`/kaggle/working`) ở cột bên phải, bấm vào dấu ba chấm cạnh từng file để tải về.

In [ ]:
print(f"{'chunking':<10} {'accuracy':>9} {'đúng':>8} {'yes':>8} {'no':>8}")
for name, report in REPORTS.items():
  print(f"{name:<10} {report['accuracy']:>9.2%} {report['correct']:>4}/{report['total']:<3} "
        f"{report['accuracy_yes']:>8.2%} {report['accuracy_no']:>8.2%}")

with open(f"{OUTPUT_DIR}/accuracy_report.json", "w", encoding="utf-8") as f:
  json.dump(REPORTS, f, indent=2)